# 🔬 LYT-Net Ablation Study
### Cross-Domain Low-Light Image Enhancement — HW5

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Run cells top to bottom
3. Results are saved to Google Drive automatically

**What this notebook does:**
Trains 7 variants of the model and evaluates each on LOLv1 / LOLv2-Real / LOLI-Street:

| Variant | D1 deeper extractor | D2 dropout | D3 learnable w |
|---------|--------------------|-----------|-----------------|
| Baseline | ✗ | ✗ | ✗ |
| +D1 | ✓ | ✗ | ✗ |
| +D2 | ✗ | ✓ | ✗ |
| +D3 | ✗ | ✗ | ✓ |
| +D1+D2 | ✓ | ✓ | ✗ |
| +D1+D3 | ✓ | ✗ | ✓ |
| +D2+D3 | ✗ | ✓ | ✓ |
| D1+D2+D3 (Ours) | ✓ | ✓ | ✓ |

## ✅ Cell 1 — Check GPU

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU found: {gpus[0]}')
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU')
print(f'TensorFlow version: {tf.__version__}')

## ✅ Cell 2 — Install dependencies

In [ ]:
!pip install -q lpips
import lpips
print('✅ lpips installed')

## ✅ Cell 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE = '/content/drive/MyDrive/LYT-Net-Ablation'
os.makedirs(DRIVE_SAVE, exist_ok=True)
print(f'✅ Results will be saved to: {DRIVE_SAVE}')

## ✅ Cell 4 — Download datasets
Downloads LOLv1, LOLv2-Real, and LOLI-Street into `/content/data/`

In [ ]:
import gdown, zipfile, os

os.makedirs('/content/data', exist_ok=True)

# ── LOLv1 ──────────────────────────────────────────────────────────────────────
# If you already have LOLv1 in Google Drive, copy it instead:
# !cp -r /content/drive/MyDrive/LOLv1 /content/data/LOLv1
print('Downloading LOLv1...')
gdown.download('https://drive.google.com/uc?id=1vhJg75hIpYvsmryyaxdygAWeHuiY_HWu',
               '/content/data/LOLv1.zip', quiet=False)
with zipfile.ZipFile('/content/data/LOLv1.zip') as z:
    z.extractall('/content/data/')
print('✅ LOLv1 ready')

# ── LOLv2-Real ─────────────────────────────────────────────────────────────────
# Uncomment if you have the file ID:
# gdown.download('https://drive.google.com/uc?id=YOUR_ID', '/content/data/LOLv2.zip')
# with zipfile.ZipFile('/content/data/LOLv2.zip') as z:
#     z.extractall('/content/data/')

# ── LOLI-Street ────────────────────────────────────────────────────────────────
# Uncomment if you have the file ID:
# gdown.download('https://drive.google.com/uc?id=YOUR_ID', '/content/data/loli.zip')
# with zipfile.ZipFile('/content/data/loli.zip') as z:
#     z.extractall('/content/data/')

print('\n📁 /content/data structure:')
for root, dirs, files in os.walk('/content/data'):
    level = root.replace('/content/data','').count(os.sep)
    if level < 3:
        print(' '*2*level + os.path.basename(root) + '/')
        if level == 2: print(' '*2*(level+1) + f'{len(files)} files')

## ✅ Cell 5 — Set dataset paths

In [ ]:
# ── Edit these paths if your data is in a different location ──────────────────
PATHS = {
    'LOLv1': {
        'train_low':  '/content/data/LOLv1/Train/input',
        'train_high': '/content/data/LOLv1/Train/target',
        'test_low':   '/content/data/LOLv1/Test/input',
        'test_high':  '/content/data/LOLv1/Test/target',
    },
    'LOLv2_Real': {
        'test_low':  '/content/data/LOLv2/Real_captured/Test/Low',
        'test_high': '/content/data/LOLv2/Real_captured/Test/Normal',
    },
    'LOLI-Street': {
        'test_low':  '/content/data/LoLI-Street/Val/low',
        'test_high': '/content/data/LoLI-Street/Val/high',
    },
}

# Verify LOLv1 (required for training)
for split, path in [('Train input', PATHS['LOLv1']['train_low']),
                     ('Train target', PATHS['LOLv1']['train_high']),
                     ('Test input',  PATHS['LOLv1']['test_low']),
                     ('Test target', PATHS['LOLv1']['test_high'])]:
    n = len(os.listdir(path)) if os.path.isdir(path) else 0
    status = '✅' if n > 0 else '❌'
    print(f'{status} LOLv1 {split}: {n} images  ({path})')

## ✅ Cell 6 — Model definitions (all 7 variants)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import keras

# ── Shared building blocks ────────────────────────────────────────────────────

class SEBlock(layers.Layer):
    def __init__(self, ch, r=16, **kw):
        super().__init__(**kw)
        self.pool = layers.GlobalAveragePooling2D()
        self.fc1  = layers.Dense(ch//r, activation='relu')
        self.fc2  = layers.Dense(ch,    activation='tanh')
    def call(self, x):
        s = tf.reshape(self.fc2(self.fc1(self.pool(x))), [-1,1,1,x.shape[-1]])
        return x * s

class MSEFBlock(layers.Layer):
    def __init__(self, filters, **kw):
        super().__init__(**kw)
        self.ln  = layers.LayerNormalization(axis=-1)
        self.dw  = layers.DepthwiseConv2D(3, padding='same')
        self.se  = SEBlock(filters)
    def call(self, x):
        n = self.ln(x)
        return layers.Add()([layers.Multiply()([self.dw(n), self.se(n)]), x])

class MHSA(layers.Layer):
    def __init__(self, d, h=4, **kw):
        super().__init__(**kw)
        self.d=d; self.h=h; self.hd=d//h
        self.Q=layers.Dense(d); self.K=layers.Dense(d)
        self.V=layers.Dense(d); self.O=layers.Dense(d)
    def call(self, x):
        bs=tf.shape(x)[0]; H=tf.shape(x)[1]; W=tf.shape(x)[2]
        def split(t): return tf.transpose(tf.reshape(t,(bs,-1,self.h,self.hd)),[0,2,1,3])
        q,k,v = split(self.Q(x)), split(self.K(x)), split(self.V(x))
        a = tf.nn.softmax(tf.matmul(q,k,transpose_b=True)/tf.math.sqrt(tf.cast(self.hd,tf.float32)),-1)
        o = self.O(tf.reshape(tf.transpose(tf.matmul(a,v),[0,2,1,3]),[bs,-1,self.d]))
        return tf.reshape(o,[bs,H,W,self.d])

def make_denoiser(num_filters, use_d2=False):
    """Factory: returns a Denoiser model with D2 on/off."""
    class Denoiser(Model):
        def __init__(self):
            super().__init__()
            k=3
            self.c1=layers.Conv2D(num_filters,k,strides=1,padding='same',activation='relu')
            self.c2=layers.Conv2D(num_filters,k,strides=2,padding='same',activation='relu')
            self.c3=layers.Conv2D(num_filters,k,strides=2,padding='same',activation='relu')
            self.c4=layers.Conv2D(num_filters,k,strides=2,padding='same',activation='relu')
            self.bn=MHSA(num_filters)
            self.dp=layers.Dropout(0.1) if use_d2 else None
            self.u2=layers.UpSampling2D(2); self.u3=layers.UpSampling2D(2); self.u4=layers.UpSampling2D(2)
            self.out=layers.Conv2D(1,k,padding='same',activation='tanh')
            self.res=layers.Conv2D(1,k,padding='same',activation='tanh')
        def call(self, inp, training=False):
            x1=self.c1(inp); x2=self.c2(x1); x3=self.c3(x2); x4=self.c4(x3)
            x=self.bn(x4)
            if self.dp is not None: x=self.dp(x, training=training)
            x=self.u4(x); x=self.u3(x3+x); x=self.u2(x2+x); x=x+x1
            return self.out(self.res(x)+inp)
    return Denoiser()

def make_lyt(filters=32, use_d1=False, use_d2=False, use_d3=False):
    """Factory: returns an LYT model with D1/D2/D3 flags."""
    class LYT(Model):
        def __init__(self):
            super().__init__()
            n = 2 if use_d1 else 1   # D1: number of conv layers
            self.py  = keras.Sequential([layers.Conv2D(filters,(3,3),activation='relu',padding='same') for _ in range(n)])
            self.pcb = keras.Sequential([layers.Conv2D(filters,(3,3),activation='relu',padding='same') for _ in range(n)])
            self.pcr = keras.Sequential([layers.Conv2D(filters,(3,3),activation='relu',padding='same') for _ in range(n)])
            self.dcb = make_denoiser(16, use_d2=use_d2)
            self.dcr = make_denoiser(16, use_d2=use_d2)
            self.lpool = layers.MaxPooling2D(8)
            self.lmhsa = MHSA(filters)
            self.lup   = layers.UpSampling2D(8)
            self.lconv = layers.Conv2D(filters,(1,1),padding='same')
            self.rconv = layers.Conv2D(filters,(1,1),padding='same')
            self.msef  = MSEFBlock(filters)
            # D3: trainable vs fixed
            self.lw = tf.Variable(0.2, trainable=use_d3, dtype=tf.float32, name='lum_w')
            self.rec  = layers.Conv2D(filters,(3,3),activation='relu',padding='same')
            self.fin  = layers.Conv2D(3,(3,3),activation='tanh',padding='same')
        def call(self, inp, training=False):
            y,cb,cr = tf.split(tf.image.rgb_to_yuv(inp),3,axis=-1)
            cb=self.dcb(cb,training=training)+cb
            cr=self.dcr(cr,training=training)+cr
            lum = self.py(y)
            ref = self.rconv(tf.concat([self.pcb(cb),self.pcr(cr)],axis=-1))
            lum = lum + self.lup(self.lmhsa(self.lpool(lum)))
            sc  = ref
            ref = self.msef(ref + self.lw * self.lconv(lum)) + sc
            return self.fin(self.rec(tf.concat([ref,lum],axis=-1)))
    return LYT()

# ── Variant registry ──────────────────────────────────────────────────────────
VARIANTS = [
    ('baseline',  dict(use_d1=False, use_d2=False, use_d3=False)),
    ('d1',        dict(use_d1=True,  use_d2=False, use_d3=False)),
    ('d2',        dict(use_d2=True,  use_d1=False, use_d3=False)),
    ('d3',        dict(use_d3=True,  use_d1=False, use_d2=False)),
    ('d1d2',      dict(use_d1=True,  use_d2=True,  use_d3=False)),
    ('d1d3',      dict(use_d1=True,  use_d2=False, use_d3=True )),
    ('d2d3',      dict(use_d1=False, use_d2=True,  use_d3=True )),
    ('d1d2d3',    dict(use_d1=True,  use_d2=True,  use_d3=True )),
]
LABELS = {
    'baseline': 'Baseline (no mod.)',
    'd1':       '+D1 only',
    'd2':       '+D2 only',
    'd3':       '+D3 only',
    'd1d2':     '+D1+D2',
    'd1d3':     '+D1+D3',
    'd2d3':     '+D2+D3',
    'd1d2d3':   'D1+D2+D3 (Ours)',
}
print('✅ All 8 model variants defined')

## ✅ Cell 7 — Loss function & data loader

In [ ]:
import tensorflow as tf
import numpy as np
import glob

# ── VGG perceptual loss ───────────────────────────────────────────────────────
def load_vgg():
    vgg = tf.keras.applications.VGG19(include_top=False, weights='imagenet')
    vgg.trainable = False
    return tf.keras.Model(vgg.input, vgg.get_layer('block3_conv3').output)

def perceptual_loss(y_true, y_pred, vgg):
    scale = tf.keras.applications.vgg19.preprocess_input
    return tf.reduce_mean(tf.square(vgg(scale(y_true*255)) - vgg(scale(y_pred*255))))

def ms_ssim_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.image.ssim_multiscale(y_true, y_pred, 1.0, power_factors=[0.5,0.5]))

def histogram_loss(y_true, y_pred, bins=64, sigma=0.02):
    edges = tf.linspace(0.0, 1.0, bins)
    def hist(x):
        h = tf.reduce_sum(tf.exp(-0.5*((tf.expand_dims(tf.reshape(x,[-1]),1)-edges)/sigma)**2), axis=0)
        return h / tf.reduce_sum(h)
    return tf.reduce_mean(tf.abs(hist(y_true) - hist(y_pred)))

def combined_loss(y_true, y_pred, vgg):
    # Rescale from [-1,1] to [0,1]
    yt = (y_true  + 1.0) / 2.0
    yp = (y_pred  + 1.0) / 2.0
    l  = (1.00 * tf.reduce_mean(tf.keras.losses.huber(yt, yp))
        + 0.06 * perceptual_loss(yt, yp, vgg)
        + 0.05 * histogram_loss(yt, yp)
        + 0.50 * ms_ssim_loss(yt, yp)
        + 0.25 * tf.reduce_mean(tf.abs(tf.reduce_mean(yt) - tf.reduce_mean(yp)))
        + 0.0083 * tf.maximum(0.0, 40.0 - tf.reduce_mean(tf.image.psnr(yt, yp, 1.0))))
    return l

# ── Data loading ──────────────────────────────────────────────────────────────
def load_image_pair(low_path, high_path, crop=256):
    def _load(lp, hp):
        l = tf.cast(tf.image.decode_image(tf.io.read_file(lp), channels=3), tf.float32) / 127.5 - 1.0
        h = tf.cast(tf.image.decode_image(tf.io.read_file(hp), channels=3), tf.float32) / 127.5 - 1.0
        # Random crop
        stacked = tf.stack([l, h], axis=0)
        cropped = tf.image.random_crop(stacked, [2, crop, crop, 3])
        return cropped[0], cropped[1]
    return _load

def make_train_ds(low_dir, high_dir, batch=4, crop=256):
    lows  = sorted(glob.glob(f'{low_dir}/*.png') + glob.glob(f'{low_dir}/*.jpg'))
    highs = sorted(glob.glob(f'{high_dir}/*.png') + glob.glob(f'{high_dir}/*.jpg'))
    assert len(lows) == len(highs) and len(lows) > 0, f'No paired images found in {low_dir}'
    ds = tf.data.Dataset.from_tensor_slices((lows, highs))
    ds = ds.shuffle(len(lows)).map(
         lambda l,h: tf.py_function(load_image_pair(l,h,crop), [l,h], [tf.float32,tf.float32]),
         num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE), len(lows)

def make_test_ds(low_dir, high_dir):
    lows  = sorted(glob.glob(f'{low_dir}/*.png') + glob.glob(f'{low_dir}/*.jpg'))
    highs = sorted(glob.glob(f'{high_dir}/*.png') + glob.glob(f'{high_dir}/*.jpg'))
    def _load(lp, hp):
        l = tf.cast(tf.image.decode_image(tf.io.read_file(lp), channels=3), tf.float32) / 127.5 - 1.0
        h = tf.cast(tf.image.decode_image(tf.io.read_file(hp), channels=3), tf.float32) / 127.5 - 1.0
        return tf.expand_dims(l,0), tf.expand_dims(h,0)
    ds = tf.data.Dataset.from_tensor_slices((lows, highs))
    return ds.map(lambda l,h: tf.py_function(_load,[l,h],[tf.float32,tf.float32]),
                  num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

print('✅ Loss function and data loader ready')

## ✅ Cell 8 — Training function
⚙️ **Adjust `EPOCHS` here** — 300 is reasonable for ablation comparison; 1000 for final model

In [ ]:
import math, datetime

EPOCHS     = 300   # ← change to 1000 for full training
BATCH_SIZE = 4
LR_INIT    = 2e-4
LR_MIN     = 1e-6

def cosine_lr(step, total_steps, t0_steps, lr0=LR_INIT, lr_min=LR_MIN, tmul=2.0):
    """Cosine annealing with warm restarts."""
    t = t0_steps
    while step >= t:
        step -= t; t = int(t * tmul)
    frac = step / t
    return lr_min + 0.5*(lr0-lr_min)*(1+math.cos(math.pi*frac))

def train_variant(variant_key, flags, vgg, epochs=EPOCHS):
    label    = LABELS[variant_key]
    save_dir = f'{DRIVE_SAVE}/{variant_key}'
    os.makedirs(save_dir, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'Training: {label}')
    print(f'{"="*60}')

    train_ds, n_train = make_train_ds(
        PATHS['LOLv1']['train_low'], PATHS['LOLv1']['train_high'],
        batch=BATCH_SIZE)
    test_ds = make_test_ds(PATHS['LOLv1']['test_low'], PATHS['LOLv1']['test_high'])

    model     = make_lyt(**flags)
    model.build(input_shape=(None,None,None,3))
    optimizer = tf.keras.optimizers.Adam(LR_INIT)

    total_steps    = epochs * (n_train // BATCH_SIZE)
    t0_steps       = 150    * (n_train // BATCH_SIZE)
    global_step    = 0
    best_psnr      = 0.0
    best_ckpt_path = None

    @tf.function
    def train_step(raw, gt):
        with tf.GradientTape() as tape:
            pred = model(raw, training=True)
            loss = combined_loss(gt, pred, vgg)
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    for epoch in range(1, epochs+1):
        ep_loss = 0.0; n_steps = 0
        for raw, gt in train_ds:
            # Update LR
            new_lr = cosine_lr(global_step, total_steps, t0_steps)
            optimizer.learning_rate.assign(new_lr)
            ep_loss += float(train_step(raw, gt))
            global_step += 1; n_steps += 1

        # Validation PSNR
        psnrs = []
        for raw, gt in test_ds:
            pred = tf.clip_by_value((model(raw, training=False)+1.0)/2.0, 0, 1)
            gt01 = (gt+1.0)/2.0
            psnrs.append(float(tf.reduce_mean(tf.image.psnr(gt01, pred, 1.0))))
        avg_psnr = float(np.mean(psnrs))

        now = datetime.datetime.now().strftime('%H:%M:%S')
        print(f'[{now}] Ep {epoch:4d}/{epochs}  loss={ep_loss/n_steps:.4f}  PSNR={avg_psnr:.2f}')

        if avg_psnr > best_psnr:
            best_psnr = avg_psnr
            best_ckpt_path = f'{save_dir}/best_ep{epoch}_psnr{avg_psnr:.2f}.weights.h5'
            model.save_weights(best_ckpt_path)
            print(f'  💾 Saved: {best_ckpt_path}')

    print(f'\n✅ {label} done. Best LOLv1 PSNR = {best_psnr:.2f} dB')
    return best_ckpt_path, model

print('✅ Training function ready')

## 🚀 Cell 9 — Run all 8 training variants
**This is the long cell — expect ~2–4 hours on T4 GPU for 300 epochs × 8 variants.**
Each variant prints its best PSNR as it goes.

In [ ]:
vgg = load_vgg()
print('✅ VGG loaded')

trained_models   = {}   # variant_key → model
trained_ckpts    = {}   # variant_key → checkpoint path

for variant_key, flags in VARIANTS:
    ckpt, model = train_variant(variant_key, flags, vgg)
    trained_models[variant_key]  = model
    trained_ckpts[variant_key]   = ckpt

print('\n🎉 All variants trained!')
for k, p in trained_ckpts.items():
    print(f'  {LABELS[k]:<25}  →  {p}')

## ✅ Cell 10 — Evaluate on all 3 test sets

In [ ]:
import torch, lpips as lpips_lib
import warnings; warnings.filterwarnings('ignore')

lpips_fn = lpips_lib.LPIPS(net='alex')

def eval_on_dataset(model, low_dir, high_dir):
    """Returns (mean_psnr, mean_ssim, mean_lpips)."""
    lows  = sorted(glob.glob(f'{low_dir}/*.png')  + glob.glob(f'{low_dir}/*.jpg'))
    highs = sorted(glob.glob(f'{high_dir}/*.png') + glob.glob(f'{high_dir}/*.jpg'))
    if not lows:
        return None
    psnrs, ssims, lpipss = [], [], []
    for lp, hp in zip(lows, highs):
        raw = tf.cast(tf.image.decode_image(tf.io.read_file(lp),channels=3),tf.float32)/127.5-1.0
        gt  = tf.cast(tf.image.decode_image(tf.io.read_file(hp),channels=3),tf.float32)/127.5-1.0
        raw = tf.expand_dims(raw,0); gt = tf.expand_dims(gt,0)
        pred = tf.clip_by_value((model(raw,training=False)+1.0)/2.0, 0, 1)
        gt01 = (gt+1.0)/2.0
        psnrs.append(float(tf.reduce_mean(tf.image.psnr(gt01, pred, 1.0))))
        ssims.append(float(tf.reduce_mean(tf.image.ssim(gt01, pred, 1.0))))
        p_pt = torch.from_numpy(pred.numpy()).permute(0,3,1,2).float()
        t_pt = torch.from_numpy(gt01.numpy()).permute(0,3,1,2).float()
        with torch.no_grad():
            lpipss.append(lpips_fn(p_pt,t_pt).item())
    return float(np.mean(psnrs)), float(np.mean(ssims)), float(np.mean(lpipss))

# ── Run evaluation for all variants ──────────────────────────────────────────
DS_EVAL = {
    'LOLv1':      (PATHS['LOLv1']['test_low'],          PATHS['LOLv1']['test_high']),
    'LOLv2_Real': (PATHS['LOLv2_Real']['test_low'],     PATHS['LOLv2_Real']['test_high']),
    'LOLI-Street':(PATHS['LOLI-Street']['test_low'],    PATHS['LOLI-Street']['test_high']),
}

RESULTS = {}   # variant_key → {ds_name: (psnr,ssim,lpips)}

for variant_key, _ in VARIANTS:
    model  = trained_models[variant_key]
    label  = LABELS[variant_key]
    print(f'\nEvaluating: {label}')
    RESULTS[variant_key] = {}
    for ds_name, (low_dir, high_dir) in DS_EVAL.items():
        r = eval_on_dataset(model, low_dir, high_dir)
        RESULTS[variant_key][ds_name] = r
        if r:
            print(f'  {ds_name:<18} PSNR={r[0]:.2f}  SSIM={r[1]:.4f}  LPIPS={r[2]:.4f}')
        else:
            print(f'  {ds_name:<18} ⚠️  dataset not found — skipped')

print('\n✅ Evaluation complete!')

## ✅ Cell 11 — Print paper-ready table and save CSV

In [ ]:
import pandas as pd

ORDER   = [k for k,_ in VARIANTS]
DS_NAMES = ['LOLv1','LOLv2_Real','LOLI-Street']

# ── Console table ─────────────────────────────────────────────────────────────
sep = '─'*108
print(f'\n{sep}')
print(f'{"Variant":<22}  {"LOLv1":^30}  {"LOLv2_Real":^30}  {"LOLI-Street":^30}')
print(f'{"":22}  {"PSNR":>8} {"SSIM":>8} {"LPIPS":>8}'
      f'  {"PSNR":>8} {"SSIM":>8} {"LPIPS":>8}'
      f'  {"PSNR":>8} {"SSIM":>8} {"LPIPS":>8}')
print(sep)

rows = []
for k in ORDER:
    label = LABELS[k]
    line  = f'{label:<22}'
    row   = {'Variant': label}
    for ds in DS_NAMES:
        r = RESULTS[k].get(ds)
        if r:
            line += f'  {r[0]:8.2f} {r[1]:8.4f} {r[2]:8.4f}'
            row[f'{ds}_PSNR'] = round(r[0],2)
            row[f'{ds}_SSIM'] = round(r[1],4)
            row[f'{ds}_LPIPS']= round(r[2],4)
        else:
            line += f'  {"N/A":>8} {"N/A":>8} {"N/A":>8}'
    print(line)
    rows.append(row)
print(sep)

# ── Save CSV ──────────────────────────────────────────────────────────────────
df = pd.DataFrame(rows)
csv_path = f'{DRIVE_SAVE}/ablation_results.csv'
df.to_csv(csv_path, index=False)
print(f'\n✅ Results saved: {csv_path}')
print('\n📋 Copy the numbers above into the LaTeX table in ablation_study_section.md')
df

## ✅ Cell 12 — Generate LaTeX table (ready to paste into paper)

In [ ]:
def make_latex(df):
    lines = []
    lines.append(r'\begin{table}[t]')
    lines.append(r'  \centering')
    lines.append(r'  \caption{Ablation study. All variants trained on LOL-v1 and evaluated')
    lines.append(r'  without fine-tuning. $\uparrow$: higher is better; $\downarrow$: lower is better.')
    lines.append(r'  \textbf{Bold}: best in column.}')
    lines.append(r'  \label{tab:ablation}')
    lines.append(r'  \setlength{\tabcolsep}{3.5pt}')
    lines.append(r'  \begin{tabular}{lccccccccc}')
    lines.append(r'    \toprule')
    lines.append(r'    & \multicolumn{3}{c}{\textbf{LOL-v1}}')
    lines.append(r'    & \multicolumn{3}{c}{\textbf{LOL-v2-Real}}')
    lines.append(r'    & \multicolumn{3}{c}{\textbf{LOLI-Street}} \\')
    lines.append(r'    \cmidrule(lr){2-4}\cmidrule(lr){5-7}\cmidrule(lr){8-10}')
    lines.append(r'    \textbf{Variant}')
    lines.append(r'      & PSNR$\uparrow$ & SSIM$\uparrow$ & LPIPS$\downarrow$')
    lines.append(r'      & PSNR$\uparrow$ & SSIM$\uparrow$ & LPIPS$\downarrow$')
    lines.append(r'      & PSNR$\uparrow$ & SSIM$\uparrow$ & LPIPS$\downarrow$ \\')
    lines.append(r'    \midrule')

    # Find best per column
    best = {}
    for ds in DS_NAMES:
        for m, fn in [('PSNR',max),('SSIM',max),('LPIPS',min)]:
            vals = [df.iloc[i][f'{ds}_{m}'] for i in range(len(df)) if f'{ds}_{m}' in df.columns and not pd.isna(df.iloc[i][f'{ds}_{m}'])]
            if vals: best[f'{ds}_{m}'] = fn(vals)

    def b(val, key):
        if pd.isna(val): return 'N/A'
        s = f'{val:.3f}' if 'SSIM' in key or 'LPIPS' in key else f'{val:.2f}'
        return f'\\textbf{{{s}}}' if abs(val - best.get(key,float('nan'))) < 1e-4 else s

    for _, row in df.iterrows():
        label = row['Variant']
        cells = [f'    {label:<25}']
        for ds in DS_NAMES:
            for m in ['PSNR','SSIM','LPIPS']:
                k = f'{ds}_{m}'
                cells.append(b(row.get(k, float('nan')), k))
        line = cells[0] + ' & ' + ' & '.join(cells[1:]) + r' \\'
        if label == 'Baseline (no mod.)':
            pass
        if label == 'D1+D2+D3 (Ours)':
            lines.append(r'    \midrule')
        lines.append(line)

    lines.append(r'    \bottomrule')
    lines.append(r'  \end{tabular}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)

latex = make_latex(df)
print(latex)

# Save to Drive
with open(f'{DRIVE_SAVE}/ablation_table.tex','w') as f:
    f.write(latex)
print(f'\n✅ LaTeX saved: {DRIVE_SAVE}/ablation_table.tex')